# Génération Key Market Data

Ce notebook génère les Key Market Data en fusionnant :
- **CSV S&P 500** : Base principale (~500 entreprises du S&P 500 avec Weight et Price)
- **CSV Performance** : Enrichissement avec métriques financières (Market Cap, Revenue, Net Income, etc.)
- **yfinance API** : Enrichissement avec secteur et industrie pour chaque entreprise

**Résultat** : `all_market_data.json` avec ~500 entreprises S&P 500 enrichies


## 1. Imports et Configuration


In [1]:
import pandas as pd
import json
import time
from pathlib import Path
from typing import Dict
import yfinance as yf
from tqdm import tqdm

# Chemins
BASE_DIR = Path('../../')
DATA_RAW = BASE_DIR / 'data' / 'raw'
DATA_GENERATED = BASE_DIR / 'data' / 'generated' / 'key_market_data'

# Créer dossier de sortie
DATA_GENERATED.mkdir(parents=True, exist_ok=True)

print(f"📁 Dossier de sortie: {DATA_GENERATED}")


📁 Dossier de sortie: ..\..\data\generated\key_market_data


## 2. Charger CSV S&P 500 (Base principale - ~500 entreprises)


In [2]:
# Charger CSV S&P 500 (contient Weight et Price pour les 503 tickers) - BASE PRINCIPALE
composition_df = pd.read_csv(DATA_RAW / '2025-08-15_composition_sp500.csv')

# Normaliser colonne Symbol -> Ticker
if 'Symbol' in composition_df.columns:
    composition_df = composition_df.rename(columns={'Symbol': 'Ticker'})

# Convertir Weight et Price (format européen avec virgule)
if 'Weight' in composition_df.columns:
    composition_df['Weight'] = composition_df['Weight'].astype(str).str.replace(',', '.').astype(float)
if 'Price' in composition_df.columns:
    composition_df['Price'] = composition_df['Price'].astype(str).str.replace(',', '.').astype(float)

print(f"✅ CSV S&P 500 chargé: {len(composition_df)} entreprises (BASE PRINCIPALE)")
print(f"   Colonnes: {list(composition_df.columns)}")
print(f"\n📊 Premières lignes:")
composition_df.head()


✅ CSV S&P 500 chargé: 503 entreprises (BASE PRINCIPALE)
   Colonnes: ['#', 'Company', 'Ticker', 'Weight', 'Price']

📊 Premières lignes:


,#,Company,Ticker,Weight,Price
0,1,Nvidia,NVDA,0.0765,181.99
1,2,Microsoft,MSFT,0.0667,520.99
2,3,"Apple Inc,",AAPL,0.0597,233.28
3,4,Amazon,AMZN,0.0427,232.14
4,5,Meta Platforms,META,0.0339,783.78


## 3. Charger CSV Performance (Enrichissement - métriques financières)


In [3]:
# Charger CSV Performance (contient toutes les métriques financières) - ENRICHISSEMENT
performance_df = pd.read_csv(DATA_RAW / '2025-09-26_stocks-performance.csv')

# Normaliser colonne Symbol -> Ticker
if 'Symbol' in performance_df.columns:
    performance_df = performance_df.rename(columns={'Symbol': 'Ticker'})

print(f"✅ CSV Performance chargé: {len(performance_df)} entreprises (ENRICHISSEMENT)")
print(f"   Colonnes: {list(performance_df.columns)}")
print(f"\n📊 Premières lignes:")
performance_df.head()


✅ CSV Performance chargé: 5508 entreprises (ENRICHISSEMENT)
   Colonnes: ['Ticker', 'Company Name', 'Market Cap', 'Revenue', 'Op. Income', 'Net Income', 'EPS', 'FCF']

📊 Premières lignes:


,Ticker,Company Name,Market Cap,Revenue,Op. Income,Net Income,EPS,FCF
0,NVDA,NVIDIA Corporation,4.338392e+12,1.652180e+11,9.598100e+10,8.659700e+10,3.52,7.202300e+10
1,MSFT,Microsoft Corporation,3.801767e+12,2.817240e+11,1.285280e+11,1.018320e+11,13.64,7.161100e+10
2,AAPL,Apple Inc.,3.791126e+12,4.086250e+11,1.302140e+11,9.928000e+10,6.59,9.618400e+10
3,GOOG,Alphabet Inc.,2.989536e+12,3.713990e+11,1.213700e+11,1.155730e+11,9.37,6.672800e+10
4,GOOGL,Alphabet Inc.,2.981796e+12,3.713990e+11,1.213700e+11,1.155730e+11,9.38,6.672800e+10


## 4. Fusionner les données (S&P 500 BASE + Performance ENRICHISSEMENT)


In [4]:
# Fusionner : S&P 500 comme BASE, enrichir avec Performance
# Méthode adaptée du collègue : merge par Ticker (left join sur S&P 500)
merged_df = pd.merge(
    composition_df,  # BASE PRINCIPALE : S&P 500 (~500 entreprises)
    performance_df[['Ticker', 'Company Name', 'Market Cap', 'Revenue', 'Op. Income', 'Net Income', 'EPS', 'FCF']],  # ENRICHISSEMENT : métriques financières
    on='Ticker',
    how='left',  # Garder tous les tickers du S&P 500, ajouter Performance si disponible
    suffixes=('', '_performance')
)

# Statistiques
total_count = len(merged_df)
performance_count = merged_df['Market Cap'].notna().sum()

print(f"✅ Fusion terminée: {total_count} entreprises (toutes dans S&P 500)")
print(f"   - Tickers avec données Performance: {performance_count}")
print(f"   - Tickers avec données S&P 500 seulement: {total_count - performance_count}")

merged_df.head()


✅ Fusion terminée: 503 entreprises (toutes dans S&P 500)
   - Tickers avec données Performance: 500
   - Tickers avec données S&P 500 seulement: 3


,#,Company,Ticker,Weight,Price,Company Name,Market Cap,Revenue,Op. Income,Net Income,EPS,FCF
0,1,Nvidia,NVDA,0.0765,181.99,NVIDIA Corporation,4.338392e+12,1.652180e+11,9.598100e+10,8.659700e+10,3.52,7.202300e+10
1,2,Microsoft,MSFT,0.0667,520.99,Microsoft Corporation,3.801767e+12,2.817240e+11,1.285280e+11,1.018320e+11,13.64,7.161100e+10
2,3,"Apple Inc,",AAPL,0.0597,233.28,Apple Inc.,3.791126e+12,4.086250e+11,1.302140e+11,9.928000e+10,6.59,9.618400e+10
3,4,Amazon,AMZN,0.0427,232.14,"Amazon.com, Inc.",2.343934e+12,6.700380e+11,7.619000e+10,7.062300e+10,6.56,5.134900e+10
4,5,Meta Platforms,META,0.0339,783.78,"Meta Platforms, Inc.",1.868445e+12,1.788040e+11,7.871000e+10,7.150700e+10,27.58,5.013700e+10


## 5. Créer structure JSON pour chaque entreprise


In [ ]:
def create_market_data_structure(row: pd.Series) -> Dict:
    """
    Crée une structure JSON enrichie pour les données de marché d'une entreprise
    Les secteurs seront ajoutés plus tard via yfinance
    """
    # Gérer company_name : priorité à Long Name (yfinance), puis Company (S&P 500), puis Company Name (performance)
    company_name = None
    if pd.notna(row.get('Long Name')):
        company_name = str(row['Long Name']).strip()
    elif pd.notna(row.get('Company')):
        company_name = str(row['Company']).strip()
    elif pd.notna(row.get('Company Name')):
        company_name = str(row['Company Name']).strip()
    
    # Toutes les entreprises sont dans S&P 500 (c'est notre base)
    is_sp500 = True  # Toujours True car on part du S&P 500
    
    # Structure de base
    data = {
        "ticker": str(row['Ticker']).upper() if pd.notna(row.get('Ticker')) else None,
        "company_name": company_name,
        "sector": str(row['Sector']) if pd.notna(row.get('Sector')) else None,
        "industry": str(row['Industry']) if pd.notna(row.get('Industry')) else None,
        "is_sp500": is_sp500,
        "sp500_weight": float(row['Weight']) if pd.notna(row.get('Weight')) else None,
        "current_price": float(row['Price']) if pd.notna(row.get('Price')) else None,
        "market_cap": float(row['Market Cap']) if pd.notna(row.get('Market Cap')) else None,
        "revenue": float(row['Revenue']) if pd.notna(row.get('Revenue')) else None,
        "operating_income": float(row['Op. Income']) if pd.notna(row.get('Op. Income')) else None,
        "net_income": float(row['Net Income']) if pd.notna(row.get('Net Income')) else None,
        "eps": float(row['EPS']) if pd.notna(row.get('EPS')) else None,
        "free_cash_flow": float(row['FCF']) if pd.notna(row.get('FCF')) else None,
    }
    
    # === CALCUL DES RATIOS FINANCIERS ===
    
    # 1. Profit Margin
    if data['revenue'] and data['net_income'] and data['revenue'] != 0:
        data['profit_margin'] = round(data['net_income'] / data['revenue'], 4)
    else:
        data['profit_margin'] = None
    
    # 2. Operating Margin
    if data['revenue'] and data['operating_income'] and data['revenue'] != 0:
        data['operating_margin'] = round(data['operating_income'] / data['revenue'], 4)
    else:
        data['operating_margin'] = None
    
    # 3. P/E Ratio (nécessite current_price)
    if data['current_price'] and data['eps'] and data['eps'] != 0:
        data['pe_ratio'] = round(data['current_price'] / data['eps'], 2)
    else:
        data['pe_ratio'] = None
    
    # 4. Price to Sales
    if data['market_cap'] and data['revenue'] and data['revenue'] != 0:
        data['price_to_sales'] = round(data['market_cap'] / data['revenue'], 2)
    else:
        data['price_to_sales'] = None
    
    # 5. FCF Margin
    if data['revenue'] and data['free_cash_flow'] and data['revenue'] != 0:
        data['fcf_margin'] = round(data['free_cash_flow'] / data['revenue'], 4)
    else:
        data['fcf_margin'] = None
    
    # 6. FCF Yield
    if data['market_cap'] and data['free_cash_flow'] and data['market_cap'] != 0:
        data['fcf_yield'] = round(data['free_cash_flow'] / data['market_cap'], 4)
    else:
        data['fcf_yield'] = None
    
    # 7. Earnings Yield
    if data['current_price'] and data['eps'] and data['current_price'] != 0:
        data['earnings_yield'] = round(data['eps'] / data['current_price'], 4)
    else:
        data['earnings_yield'] = None
    
    # 8. Market Cap to Earnings
    if data['market_cap'] and data['net_income'] and data['net_income'] != 0:
        data['market_cap_to_earnings'] = round(data['market_cap'] / data['net_income'], 2)
    else:
        data['market_cap_to_earnings'] = None
    
    # 9. Operating Efficiency
    if data['operating_income'] and data['net_income'] and data['net_income'] != 0:
        data['operating_efficiency'] = round(data['operating_income'] / data['net_income'], 2)
    else:
        data['operating_efficiency'] = None
    
    # 10. Cash Conversion
    if data['net_income'] and data['free_cash_flow'] and data['net_income'] != 0:
        data['cash_conversion'] = round(data['free_cash_flow'] / data['net_income'], 2)
    else:
        data['cash_conversion'] = None
    
    return data

# Test sur une ligne
if len(merged_df) > 0:
    example = create_market_data_structure(merged_df.iloc[0])
    print("✅ Exemple de structure générée:")
    print(json.dumps(example, indent=2, ensure_ascii=False))


## 6. Générer all_market_data.json


In [ ]:
# Créer dictionnaire pour toutes les entreprises (sans secteurs pour l'instant)
all_market_data = {}

for idx, row in merged_df.iterrows():
    ticker = str(row['Ticker']).upper() if pd.notna(row.get('Ticker')) else None
    if ticker:
        all_market_data[ticker] = create_market_data_structure(row)

print(f"✅ Structures créées: {len(all_market_data)} entreprises")

# Sauvegarder en JSON
output_file = DATA_GENERATED / "all_market_data.json"
with open(output_file, 'w', encoding='utf-8') as f:
    json.dump(all_market_data, f, indent=2, ensure_ascii=False)

print(f"✅ Fichier JSON sauvegardé: {output_file}")
print(f"   Chemin absolu: {output_file.resolve()}")

# Statistiques finales
performance_count = sum(1 for v in all_market_data.values() if v.get('market_cap') is not None)

print(f"\n📊 Statistiques:")
print(f"   - Total entreprises S&P 500: {len(all_market_data)}")
print(f"   - Tickers avec données Performance: {performance_count}")
print(f"   - Tickers avec prix (S&P 500): {sum(1 for v in all_market_data.values() if v.get('current_price'))}")
print(f"   - Tickers avec company_name: {sum(1 for v in all_market_data.values() if v.get('company_name'))}")
print(f"\n⚠️ Note: Les secteurs seront ajoutés à l'étape suivante (yfinance)")


✅ Exemple de structure générée (avec secteur):
{
  "ticker": "NVDA",
  "company_name": "Nvidia",
  "sector": null,
  "industry": null,
  "is_sp500": true,
  "sp500_weight": 0.0765,
  "current_price": 181.99,
  "market_cap": 4338391930000.0,
  "revenue": 165218000000.0,
  "operating_income": 95981000000.0,
  "net_income": 86597000000.0,
  "eps": 3.52,
  "free_cash_flow": 72023000000.0,
  "profit_margin": 0.5241,
  "operating_margin": 0.5809,
  "pe_ratio": 51.7,
  "price_to_sales": 26.26,
  "fcf_margin": 0.4359,
  "fcf_yield": 0.0166,
  "earnings_yield": 0.0193,
  "market_cap_to_earnings": 50.1,
  "operating_efficiency": 1.11,
  "cash_conversion": 0.83
}


## 7. Sauvegarder aussi en CSV


In [ ]:
# Sauvegarder le DataFrame merged en CSV pour visualisation
csv_file = DATA_GENERATED / "merged_market_data.csv"
merged_df.to_csv(csv_file, index=False, encoding='utf-8')
print(f"✅ CSV sauvegardé: {csv_file}")
print(f"   Chemin absolu: {csv_file.resolve()}")
print(f"   Lignes: {len(merged_df)}")
print(f"   Colonnes: {len(merged_df.columns)}")

# Afficher les colonnes
print(f"\n📋 Colonnes dans le CSV:")
print(f"   {', '.join(merged_df.columns.tolist())}")

print(f"\n⚠️ Note: Les secteurs seront ajoutés à l'étape suivante (yfinance)")


⚠️ Les secteurs n'ont pas encore été récupérés via yfinance.
   Les fichiers seront sauvegardés sans secteurs.
   Vous pourrez les ajouter plus tard en réexécutant la cellule 10 puis cette cellule.

✅ Structures créées: 503 entreprises
✅ Fichier JSON sauvegardé: ..\..\data\generated\key_market_data\all_market_data.json
   Chemin absolu: C:\Users\rayya\Desktop\Datathon2025\data\generated\key_market_data\all_market_data.json

📊 Statistiques:
   - Total entreprises S&P 500: 503
   - Tickers avec données Performance: 500
   - Tickers avec prix (S&P 500): 503
   - Tickers avec company_name: 503


## 7.5. Nettoyer les données (supprimer lignes/entrées avec données vides)

In [ ]:
# ============================================
# NETTOYAGE DES DONNÉES
# ============================================
# Supprimer les lignes/entrées avec des données manquantes importantes

print("🧹 Nettoyage des données...")

# --- NETTOYAGE CSV ---
print("\n📊 Nettoyage du CSV (merged_df):")
print(f"   Lignes avant nettoyage: {len(merged_df)}")

# Supprimer les lignes où le Ticker est manquant (critère minimum)
merged_df_cleaned = merged_df.dropna(subset=['Ticker']).copy()

# Optionnel : Supprimer les lignes où toutes les données financières sont manquantes
# Garder au moins une donnée : Market Cap, Revenue, ou Net Income
merged_df_cleaned = merged_df_cleaned[
    merged_df_cleaned[['Market Cap', 'Revenue', 'Net Income']].notna().any(axis=1)
]

print(f"   Lignes après nettoyage: {len(merged_df_cleaned)}")
print(f"   Lignes supprimées: {len(merged_df) - len(merged_df_cleaned)}")

# Mettre à jour merged_df
merged_df = merged_df_cleaned.copy()

# Resauvegarder le CSV nettoyé
csv_file = DATA_GENERATED / "merged_market_data.csv"
merged_df.to_csv(csv_file, index=False, encoding='utf-8')
print(f"\n✅ CSV nettoyé sauvegardé: {csv_file}")

# --- NETTOYAGE JSON ---
print("\n📄 Nettoyage du JSON (all_market_data):")
print(f"   Entreprises avant nettoyage: {len(all_market_data)}")

# Filtrer les entrées : garder seulement celles avec au moins une donnée financière
all_market_data_cleaned = {}
for ticker, data in all_market_data.items():
    # Garder si au moins une de ces données existe
    if (data.get('market_cap') is not None or 
        data.get('revenue') is not None or 
        data.get('net_income') is not None):
        all_market_data_cleaned[ticker] = data

print(f"   Entreprises après nettoyage: {len(all_market_data_cleaned)}")
print(f"   Entreprises supprimées: {len(all_market_data) - len(all_market_data_cleaned)}")

# Mettre à jour all_market_data
all_market_data = all_market_data_cleaned.copy()

# Resauvegarder le JSON nettoyé
output_file = DATA_GENERATED / "all_market_data.json"
with open(output_file, 'w', encoding='utf-8') as f:
    json.dump(all_market_data, f, indent=2, ensure_ascii=False)

print(f"\n✅ JSON nettoyé sauvegardé: {output_file}")

# Statistiques finales après nettoyage
print(f"\n📊 Statistiques après nettoyage:")
print(f"   - CSV: {len(merged_df)} entreprises")
print(f"   - JSON: {len(all_market_data)} entreprises")
print(f"   - Données complètes: {merged_df[['Market Cap', 'Revenue', 'Net Income']].notna().all(axis=1).sum()} entreprises")

## 8. Récupérer les secteurs via yfinance (ENRICHISSEMENT OPTIONNEL)

**Note** : Cette étape peut prendre du temps (rate limiting API). On récupère le secteur et l'industrie pour chaque ticker.  
**Les fichiers JSON et CSV ont déjà été générés précédemment. Cette étape permet de les enrichir avec les secteurs.**


In [ ]:
# Préparer les symboles pour Yahoo Finance
# Remplacer les virgules par des tirets (ex: BRK,B → BRK-B) pour yfinance
def normalize_ticker_for_yfinance(ticker: str) -> str:
    """Normalise le ticker pour yfinance (remplace virgules par tirets)"""
    if pd.isna(ticker):
        return None
    return str(ticker).replace(",", "-")

# Récupérer les secteurs et industries via yfinance
# Code adapté du collègue avec gestion d'erreurs et rate limiting
sector_dict = {}
industry_dict = {}
company_long_name_dict = {}

print("🔍 Récupération des secteurs et industries via yfinance...")
print(f"   Total tickers à traiter: {len(merged_df)}")
print(f"   ⚠️ Cette étape peut prendre du temps (rate limiting API)\n")

# Limiter aux tickers uniques pour éviter les doublons
# On travaille uniquement sur les ~500 entreprises du S&P 500
unique_tickers = merged_df['Ticker'].dropna().unique()
print(f"   Tickers S&P 500 uniques: {len(unique_tickers)}")

# Option pour limiter le nombre de tickers (pour test rapide)
# LIMIT_TICKERS = 100  # Décommenter pour tester avec seulement 100 tickers
LIMIT_TICKERS = None  # None = tous les tickers

if LIMIT_TICKERS:
    unique_tickers = unique_tickers[:LIMIT_TICKERS]
    print(f"   ⚠️ Mode TEST: Limitation à {LIMIT_TICKERS} tickers")

for ticker in tqdm(unique_tickers, desc="Récupération secteurs"):
    try:
        # Normaliser le ticker pour yfinance
        yf_ticker = normalize_ticker_for_yfinance(ticker)
        if not yf_ticker:
            continue
            
        # Récupérer les infos via yfinance
        company = yf.Ticker(yf_ticker)
        info = company.info  # Dictionnaire avec toutes les infos
        
        # Extraire secteur, industrie et nom long
        sector_dict[ticker] = info.get("sector", "Unknown")
        industry_dict[ticker] = info.get("industry", "Unknown")
        company_long_name_dict[ticker] = info.get("longName", None)
        
        # Délai pour éviter rate limiting (0.5 secondes entre chaque appel)
        time.sleep(0.5)
        
    except Exception as e:
        # Si erreur, mettre "Unknown"
        sector_dict[ticker] = "Unknown"
        industry_dict[ticker] = "Unknown"
        company_long_name_dict[ticker] = None
        # print(f"   ⚠️ Erreur pour {ticker}: {str(e)}")

print("\n✅ Récupération terminée !")

# Ajouter les colonnes au DataFrame
merged_df['Sector'] = merged_df['Ticker'].map(sector_dict)
merged_df['Industry'] = merged_df['Ticker'].map(industry_dict)
merged_df['Long Name'] = merged_df['Ticker'].map(company_long_name_dict)

# Statistiques
print(f"\n📊 Statistiques des secteurs:")
sector_counts = merged_df['Sector'].value_counts()
print(sector_counts.head(10))

print(f"\n✅ Colonnes ajoutées: Sector, Industry, Long Name")
print(f"\n📊 Aperçu:")
merged_df[['Ticker', 'Company', 'Sector', 'Industry']].head(20)


## 9. Régénérer les fichiers avec les secteurs (après enrichissement yfinance)


In [ ]:
# Régénérer all_market_data.json avec les secteurs
print("🔄 Régénération de all_market_data.json avec les secteurs...")

all_market_data_enriched = {}
for idx, row in merged_df.iterrows():
    ticker = str(row['Ticker']).upper() if pd.notna(row.get('Ticker')) else None
    if ticker:
        all_market_data_enriched[ticker] = create_market_data_structure(row)

# Sauvegarder JSON avec secteurs
output_file = DATA_GENERATED / "all_market_data.json"
with open(output_file, 'w', encoding='utf-8') as f:
    json.dump(all_market_data_enriched, f, indent=2, ensure_ascii=False)

print(f"✅ Fichier JSON mis à jour avec secteurs: {output_file}")

# Sauvegarder CSV avec secteurs
csv_file = DATA_GENERATED / "merged_market_data.csv"
merged_df.to_csv(csv_file, index=False, encoding='utf-8')
print(f"✅ Fichier CSV mis à jour avec secteurs: {csv_file}")

# Statistiques finales
sector_count = sum(1 for v in all_market_data_enriched.values() if v.get('sector') and v.get('sector') != 'Unknown')
print(f"\n📊 Statistiques finales:")
print(f"   - Total entreprises S&P 500: {len(all_market_data_enriched)}")
print(f"   - Tickers avec secteur: {sector_count}")
print(f"   - Colonnes ajoutées au CSV: Sector, Industry, Long Name")


✅ CSV sauvegardé: ..\..\data\generated\key_market_data\merged_market_data.csv
   Chemin absolu: C:\Users\rayya\Desktop\Datathon2025\data\generated\key_market_data\merged_market_data.csv
   Lignes: 503
   Colonnes: 12

📋 Colonnes dans le CSV:
   #, Company, Ticker, Weight, Price, Company Name, Market Cap, Revenue, Op. Income, Net Income, EPS, FCF

⚠️ Secteurs non encore récupérés. Réexécutez la cellule 10 pour les ajouter.


## 10. Exemples : Afficher quelques entreprises (avec secteurs)


In [ ]:
# Exemples S&P 500
print("📊 Exemple S&P 500 (AAPL):")
if 'AAPL' in all_market_data_enriched:
    print(json.dumps(all_market_data_enriched['AAPL'], indent=2, ensure_ascii=False))

print("\n" + "="*80 + "\n")

# Exemple d'une autre entreprise S&P 500
print("📊 Exemple S&P 500 (MSFT):")
if 'MSFT' in all_market_data_enriched:
    print(json.dumps(all_market_data_enriched['MSFT'], indent=2, ensure_ascii=False))

print("\n" + "="*80 + "\n")

# Résumé par secteur (comme dans le code du collègue)
print("📊 Résumé par secteur (Top 10):")
if 'Sector' in merged_df.columns:
    grouped = merged_df.groupby('Sector')
    for sector, df in list(grouped)[:10]:
        total_weight = df['Weight'].sum()
        print(f"\n=== {sector} ===")
        print(f"   Total entreprises S&P 500: {len(df)}")
        print(f"   Poids total dans S&P 500: {total_weight:.4f} ({total_weight*100:.2f}%)")
        print(f"   Exemples: {', '.join(df.head(5)['Ticker'].tolist())}")
